In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load the three datasets
print("Loading datasets...")
try:
    train_df = pd.read_csv(MACHINE_A_TRAIN_SET_FILE)
    val_df   = pd.read_csv(MACHINE_A_VAL_SET_FILE)
    test_df  = pd.read_csv(MACHINE_A_TEST_SET_FILE)
    
    # Label them for plotting
    train_df['dataset'] = 'Train'
    val_df['dataset']   = 'Validation'
    test_df['dataset']  = 'Test'
    
    # Combine for easy plotting
    combined_df = pd.concat([train_df, val_df, test_df], axis=0)
    
    print(f"Loaded successfully.")
    print(f"Train: {train_df.shape}")
    print(f"Val:   {val_df.shape}")
    print(f"Test:  {test_df.shape}")

except FileNotFoundError as e:
    print(f"ERROR: Could not find split files. Did you run Notebook 22? \n{e}")

In [ ]:
# Visual Distribution Check (KDE Plots)
# We want to see if the curves overlap. 
# If one curve is shifted far away, that split is biased.

# Identify numeric features (excluding labels/metadata)
numeric_cols = [c for c in train_df.columns 
                if c not in ['label', 'dataset', 'filename', 'source', 'jammer_type'] 
                and np.issubdtype(train_df[c].dtype, np.number)]

print(f"Verifying Distributions for: {numeric_cols}")

palette = {'Train': 'C0', 'Validation': 'C1', 'Test': 'C2'}

num_features = len(numeric_cols)
cols = 2
rows = (num_features + 1) // cols

plt.figure(figsize=(15, 4 * rows))

for i, feature in enumerate(numeric_cols):
    plt.subplot(rows, cols, i + 1)
    
    sns.kdeplot(
        data=combined_df, 
        x=feature, 
        hue='dataset', 
        palette=palette,
        fill=True, 
        alpha=0.1,      # Transparent fill to see overlaps
        linewidth=2
    )
    
    plt.title(f"{feature} Distribution")
    plt.ylabel("Density")

plt.tight_layout()
save_plot("01_split_distribution_check", prefix="23")
plt.show()

In [ ]:
# Statistical Health Check (The "Table of Truth")
# Visuals can be deceptive. Let's check the raw numbers.

stats = []

for feature in numeric_cols:
    # Calculate Mean and Std for each split
    t_mean, t_std = train_df[feature].mean(), train_df[feature].std()
    v_mean, v_std = val_df[feature].mean(), val_df[feature].std()
    s_mean, s_std = test_df[feature].mean(), test_df[feature].std()
    
    # Check difference (Train vs Test) relative to standard deviation
    # Z-score diff: |Mean1 - Mean2| / Avg_Std
    diff_metric = abs(t_mean - s_mean) / ((t_std + s_std) / 2)
    
    status = "OK" if diff_metric < 0.1 else "DRIFT"
    
    stats.append({
        'Feature': feature,
        'Train Mean': f"{t_mean:.4f}",
        'Test Mean':  f"{s_mean:.4f}",
        'Delta (Z)':  f"{diff_metric:.4f}",
        'Status': status
    })

stats_df = pd.DataFrame(stats)
print("--- Statistical Consistency Check ---")
print("Delta (Z) should be close to 0.0. If > 0.1, check random seed.")
stats_df

In [ ]:
# Class Balance Check
# Ensure the ratio of Clean/Jamming is identical across splits

def get_ratio(df):
    counts = df['label'].value_counts(normalize=True)
    return counts.get(1, 0) # Percentage of Jamming (Label 1)

ratios = {
    'Train': get_ratio(train_df),
    'Val':   get_ratio(val_df),
    'Test':  get_ratio(test_df)
}

plt.figure(figsize=(8, 4))
bars = plt.bar(ratios.keys(), ratios.values(), color=['C0', 'C1', 'C2'])

# Add text
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{height:.1%}', ha='center', va='bottom')

plt.title("Jamming Ratio across Splits (Should be equal)")
plt.ylim(0, 1.0)
plt.ylabel("Percentage of Jamming Samples")
save_plot("02_split_balance_check", prefix="23")
plt.show()

## Decision: Dataset Splitting & Verification

**Objective:**
To generate static, reproducible subsets (Train, Validation, Test) using stratified sampling and to verify statistical consistency across splits to prevent covariate shift or prior probability bias.

**Results:**
The dataset was split into **Train (70%)**, **Validation (15%)**, and **Test (15%)**. A rigorous audit yielded the following:

* **Class Balance:** The ratio of Jamming signals was preserved perfectly at **66.7%** across all three sets (Train: `66.7%`, Val: `66.7%`, Test: `66.7%`).
* **Distribution Consistency:** The statistical difference (Delta Z-Score) between Train and Test features was negligible across the board:
* *Mean Power:* `0.0009`
* *Kurtosis:* `0.0034`
* *Max Delta Observed:* `0.0065` (Spectral Flatness) - well below the drift threshold of `0.1`.



**Conclusion:**
The three splits are statistically indistinguishable.

* **Decision:** We will proceed to Model Training using these static CSV files (`train_set.csv`, `val_set.csv`, `test_set.csv`).
* **Rationale:** The stratified split successfully maintained the 2:1 class imbalance, ensuring the model learns the correct prior probabilities. The near-zero Z-scores confirm that the Test set is a valid, unbiased representation of the Training physics, eliminating concerns about data leakage or distribution bias.
